### Dataset Collection

- Scrape product reviews from Flipkart or Amazon (minimum 100 reviews)

- You can use Selenium, BeautifulSoup, or any reliable scraping method

- Store data in CSV format with at least one column: review_text

- Ensure data is clean and readable

- Do not scrape restricted or sensitive content

In [2]:
import requests
from bs4 import BeautifulSoup as bs

In [3]:
def scrape(url):
    headers = {
        'authority': 'www.amazon.com',
        'pragma': 'no-cache',
        'cache-control': 'no-cache',
        'dnt': '1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'sec-fetch-site': 'none',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-dest': 'document',
        'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8',
    }

    # Download the page using requests
    print("Downloading %s"%url)
    r = requests.get(url, headers=headers)

    
    # Simple check to check if page was blocked (Usually 503)
    if r.status_code > 500:
        if "To discuss automated access to Amazon data please contact" in r.text:
            print("Page %s was blocked by Amazon. Please try using better proxies\n"%url)
        else:
            print("Page %s must have been blocked by Amazon as the status code was %d"%(url,r.status_code))
        return None
    # Pass the HTML of the page and create
    return r.text

In [4]:
urls = ["https://www.amazon.com/s?k=tv","https://www.amazon.in/s?k=mobile", "https://www.amazon.in/s?k=earbuds+buds", "https://www.amazon.in/s?k=earbuds+samsung"]

In [6]:
extracted_links = []
for url in urls:
  page = scrape(url)
  page_html = bs(page,"html.parser")
  per_product = page_html.find_all("div" ,{"class" :"rush-component"})
  a_tags = page_html.find_all("a", class_="a-link-normal")
  print(f"URL: {url} Count: {len(a_tags)}")
  if a_tags:
      for tag in a_tags:
          link = tag.get("href")
          if link:
              extracted_links.append(link)

URL: https://www.amazon.com/s?k=tv Count: 406
URL: https://www.amazon.in/s?k=mobile Count: 422
URL: https://www.amazon.in/s?k=earbuds+buds Count: 495
URL: https://www.amazon.in/s?k=earbuds+samsung Count: 399


In [7]:
product_links = [a for a in extracted_links if a.find('#customerReviews') > 0 ]

In [8]:
len(product_links)

132

In [9]:
reviews =[]
for url in product_links:
  try:
    full_url = url if url.startswith('https:') else 'https://www.amazon.in' + url
    page = scrape(full_url)
    soup = bs(page,"html.parser")
    div_reviews = soup.find_all('div', {'class': 'a-section celwidget'})
    for review_div in div_reviews:
      star_rating_span = review_div.find('i', {'data-hook': 'review-star-rating'})
      star_rating = star_rating_span.find('span', class_='a-icon-alt').get_text(strip=True) if star_rating_span else 'N/A'

      review_title_span = review_div.find('a', {'data-hook': 'review-title'})
      review_title = review_title_span.find_all('span')[-1].get_text(strip=True) if review_title_span and review_title_span.find_all('span') else 'N/A'

      review_body_span = review_div.find('span', {'data-hook': 'review-body'})
      review_description = review_body_span.get_text(strip=True) if review_body_span else 'N/A'
      if star_rating != 'N/A' and review_title != 'N/A' and review_description != 'N/A':
        reviews.append((star_rating,review_title, review_description))
  except Exception as e:
    print(f"Error processing URL: {url}")

In [10]:
import csv
import os

output_filename = 'data\\amazon_reviews.csv'

with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile)
    # Write the header row
    csv_writer.writerow(['Star Rating', 'Review Title', 'Review Description'])
    # Write the review data
    csv_writer.writerows(reviews)

print(f"Successfully saved {len(reviews)} reviews to {output_filename}")

Successfully saved 724 reviews to data\amazon_reviews.csv
